#Data Quality and Validationin ETL

**Question 1: Define Data Quality in the context of ETL pipelines. Why is it more than just data cleaning?**

Data Quality refers to the accuracy, completeness, consistency, validity, uniqueness, and reliability of data throughout the ETL (Extract, Transform, Load) process.

Data quality is more than just data cleaning because it also includes:

* Ensuring data is accurate and complete.
* Validating data against business rules.
* Removing duplicate records.
* Maintaining consistency across different data sources.
* Preserving referential integrity.
* Ensuring data is suitable for reporting and analytics.

Thus, data cleaning is only one part of achieving high-quality data.

**Question 2: Explain why poor data quality leads to misleading dashboards and incorrect decisions**

Poor data quality affects business decisions because:

* Incorrect or missing values produce inaccurate reports.
* Duplicate records inflate sales or customer counts.
* Inconsistent data causes reporting errors.
* Invalid data reduces trust in dashboards.
* Decision-makers may take wrong actions based on incorrect information.

Example: If duplicate sales transactions exist, the dashboard may show higher revenue than actually earned.

**Question 3: What is duplicate data? Explain three causes in ETL pipelines.**

Duplicate data means the same record appears more than once in a dataset.

Three common causes:
1. Repeated data extraction from the same source.
2. Improper table joins during ETL transformation.
3. Multiple system integrations importing the same record.

Duplicate data increases storage, reduces data quality, and leads to incorrect analytics.

**Question 4: Differentiate between exact, partial, and fuzzy duplicates.**

| Exact Duplicate                         | Partial Duplicate                          | Fuzzy Duplicate                               |
| --------------------------------------- | ------------------------------------------ | --------------------------------------------- |
| All values are identical.               | Some fields are identical.                 | Records are similar but not exactly the same. |
| Easy to detect.                         | Requires column comparison.                | Uses similarity matching algorithms.          |
| Example: Same customer record repeated. | Same customer with different phone number. | "Rahul Mehta" and "Rahul M."                  |


**Question 5: Why should data validation be performed during transformation rather than after loading?**

Data validation should be performed during transformation because:

* Errors are detected before loading into the target database.
* Invalid data is prevented from entering the data warehouse.
* ETL performance improves by reducing rework.
* Reports and dashboards remain accurate.
* It saves time and storage costs.

**Question 6: Explain how business rules help in validating data accuracy. Give an example.**

Business rules define conditions that data must satisfy before loading.

Examples of business rules include:

* Quantity must be greater than zero.
* Transaction amount cannot be negative.
* Customer ID must exist in the customer master table.
* Transaction date cannot be blank.

Example:

If Quantity = NULL, the ETL process rejects or corrects the record because quantity is mandatory.

**Question 7: SQL Query to Find Duplicate Business Keys**

In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")

data = {
    "Txn_ID":[201,202,203,204,205,206,207,208],
    "Customer_ID":["C101","C102","C101","C103","C104","C105","C106","C101"],
    "Customer_Name":["Rahul Mehta","Anjali Rao","Rahul Mehta","Suresh Iyer",
                     "Neha Singh","N/A","Amit Verma","Rahul Mehta"],
    "Product_ID":["P11","P12","P11","P13","P14","P15","P16","P11"],
    "Quantity":[2,1,2,3,None,1,1,2],
    "Txn_Amount":[4000,1500,4000,6000,2500,None,1800,4000],
    "Txn_Date":["2025-12-01","2025-12-01","2025-12-01","2025-12-02",
                "2025-12-02","2025-12-03",None,"2025-12-01"],
    "City":["Mumbai","Bengaluru","Mumbai","Chennai",
            "Delhi","Pune","Pune","Mumbai"]
}

df = pd.DataFrame(data)

df.to_sql("Sales_Transactions", conn, index=False, if_exists="replace")

8

**Question 8: SQL Query to Detect Referential Integrity Violations**

In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")

# Customers_Master Table
customers = pd.DataFrame({
    "CustomerID":["C101","C102","C103","C104"],
    "CustomerName":["Rahul Mehta","Anjali Rao","Suresh Iyer","Neha Singh"],
    "City":["Mumbai","Bengaluru","Chennai","Delhi"]
})

customers.to_sql("Customers_Master", conn, index=False, if_exists="replace")

# Sales_Transactions Table
sales = pd.DataFrame({
    "Txn_ID":[201,202,203,204,205,206,207,208],
    "Customer_ID":["C101","C102","C101","C103","C104","C105","C106","C101"],
    "Customer_Name":["Rahul Mehta","Anjali Rao","Rahul Mehta","Suresh Iyer",
                     "Neha Singh","N/A","Amit Verma","Rahul Mehta"],
    "Product_ID":["P11","P12","P11","P13","P14","P15","P16","P11"],
    "Quantity":[2,1,2,3,None,1,1,2],
    "Txn_Amount":[4000,1500,4000,6000,2500,None,1800,4000],
    "Txn_Date":["2025-12-01","2025-12-01","2025-12-01",
                "2025-12-02","2025-12-02",
                "2025-12-03",None,"2025-12-01"],
    "City":["Mumbai","Bengaluru","Mumbai","Chennai",
            "Delhi","Pune","Pune","Mumbai"]
})

sales.to_sql("Sales_Transactions", conn, index=False, if_exists="replace")

print("Tables Created Successfully")

Tables Created Successfully


In [9]:
query = """
SELECT
    s.Customer_ID,
    s.Customer_Name
FROM Sales_Transactions s
LEFT JOIN Customers_Master c
ON s.Customer_ID = c.CustomerID
WHERE c.CustomerID IS NULL;
"""

result = pd.read_sql_query(query, conn)

print(result)

  Customer_ID Customer_Name
0        C105           N/A
1        C106    Amit Verma
